# Socratic Tutor Swarm — multi-agent tutoring on the Actor runtime

A tutor that never just tells you the answer. Built on `vidbyte-sdk`'s
**`ActorRuntime`**: a coordinator agent delegates each turn to specialized
sub-actors running as asynchronous message-passing actors — explainer,
question-writer, misconception-checker — then synthesizes their work into one
Socratic response.

Key runtime ideas on display:

- **Swapping the execution loop is one constructor argument.** The same
  `BaseAgent` class runs linear by default; here it runs an async actor swarm.
- **`dynamic_actors=True`** — the coordinator spins up the sub-actors it needs
  for the turn rather than using a fixed pipeline.
- **`termination_mode="quiescence"`** — the turn ends when the actor system
  goes quiet, not when a fixed step count runs out.
- **Cheap workers, smart coordinator** — `worker_model` lets sub-actors run on
  a faster, cheaper model than the coordinator that synthesizes them.

Before running: `cp .env.example .env` and add your provider key.

In [ ]:
import os

from dotenv import load_dotenv
from vidbyte import BaseAgent
from vidbyte.agents.runtimes.configs import ActorRuntime

load_dotenv()

SYSTEM_PROMPT = """You are the coordinator of a Socratic tutoring swarm.

For every learner message, delegate to your sub-actors:
- explainer: produce the cleanest correct explanation of the concept at hand.
  You hold this in reserve to calibrate hints — you do NOT recite it.
- question-writer: draft the next probing question, pitched just beyond what
  the learner has already demonstrated they understand.
- misconception-checker: examine the learner's last answer for specific,
  named misconceptions (not generic wrongness).

Then synthesize ONE reply that:
1. Briefly affirms exactly what was right in the learner's answer (if anything).
2. Surfaces at most one misconception, by implication or counterexample —
   never by lecture.
3. Ends with a single probing question.

Hard rules: never give the full answer while the learner is making progress.
If the learner is stuck twice in a row on the same point, give the smallest
hint that unsticks them. Keep replies under 120 words."""

tutor = BaseAgent(
    name="tutor-coordinator",
    system_prompt=SYSTEM_PROMPT,
    provider=os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai"),
    model_name=os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1"),
    runtime=ActorRuntime(
        dynamic_actors=True,
        worker_model=os.getenv("VIDBYTE_COOKBOOK_WORKER_MODEL", "gpt-4o-mini"),
        max_loop=30,
        termination_mode="quiescence",
    ),
)

## Ask your first question

Each cell below is one turn. The agent's history persists across `run()`
calls, so the tutor remembers what you've already demonstrated.

In [ ]:
reply = tutor.run("Can you explain why spaced repetition works?")
print(reply.content)

## Keep the dialogue going

Edit `my_reply` with your honest attempt and re-run for each turn. Watch how
the tutor affirms what's right, surfaces at most one misconception, and always
ends with a question — that contract comes from the coordinator's synthesis
over the three sub-actors.

In [ ]:
my_reply = "Maybe because it's harder, so your brain works more?"

reply = tutor.run(my_reply)
print(reply.content)

## Adapt it

- Add a `learner-state` actor that queries the Vidbyte API (see
  [`../study-agent`](../study-agent/)) so probing questions target the
  learner's actual gap map.
- Switch `topology` to broadcast for a panel-discussion dynamic.
- Raise `max_loop` and lower `worker_model` cost for deeper deliberation per turn.